# Descriptive statistics table for treated and control municipalities (fixed)

This notebook loads the **fixed** analysis-ready municipality-quarter panel from `ROOT/out`, constructs a pooled **treated vs control** grouping, and creates a descriptive-statistics table with these rows:

- Average Rent Price Pre
- Average Rent Price Post
- Yearly Rent Growth (2011-2025)
- Yearly Rent Growth (2022-2024)
- Average Rent Growth Post
- Tenancy Agreements Pre
- Tenancy Agreements Post
- Rent Price/Taxable Base
- Yearly Taxable Base
- Population
- Natives
- Foreigners
- Net Inflows
- Net Outflows
- Number of Municipalities
- Municipalities with Data on Rents

**Important choice:** for this pooled descriptive table, `Pre`/`Post` are defined relative to the **first implementation quarter**, with:

- `Pre`: before **2024-Q2**
- `Post`: from **2024-Q2** onward

This is a descriptive convention for the table, not the staggered-treatment estimator itself.

**Fixes in this version:**
- Prefer `analysis_ready_panel_fixed.*` over the older file.
- Respect the new rent/contract exclusion flags from the corrected pipeline.
- Use only **valid rent series** for rent-based rows.
- Avoid contaminated price/growth values from municipalities without true rent data.
- Count `Municipalities with Data on Rents` using the municipality-level observed-price flag where available.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [2]:

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    OUT_DIR / "analysis_ready_panel_fixed.pkl",
    OUT_DIR / "analysis_ready_panel_fixed.jsonl",
    OUT_DIR / "analysis_ready_panel.pkl",
    OUT_DIR / "analysis_ready_panel.jsonl",
]

panel_path = next((p for p in candidate_paths if p.exists()), None)
if panel_path is None:
    raise FileNotFoundError(
        "Could not find any analysis-ready panel in ROOT/out. "
        f"Tried: {[p.name for p in candidate_paths]}"
    )

if panel_path.suffix == ".pkl":
    panel = pd.read_pickle(panel_path)
elif panel_path.suffix == ".jsonl":
    panel = pd.read_json(panel_path, orient="records", lines=True)
else:
    panel = pd.read_csv(panel_path)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("Loaded from:", panel_path)
print("Shape:", panel.shape)
display(panel.head())


ROOT: /home/venatio/documents/university/public-sector-2026
OUT_DIR: /home/venatio/documents/university/public-sector-2026/out
Loaded from: /home/venatio/documents/university/public-sector-2026/out/analysis_ready_panel.pkl
Shape: (56908, 61)


,Code,Municipality,municipality_norm,Year,Quarter,quarter_num,time_index,Prices,Contracts,price_observed_raw,contracts_observed_raw,price_imputed,contracts_imputed,has_any_observed_price,has_any_observed_contracts,exclude_from_price_analysis,exclude_from_contract_analysis,has_valid_price_for_analysis,has_valid_contracts_for_analysis,Treated 1,Treated 2,Control,ever_treated,cohort,first_treat_period,first_treat_index,treated,event_time,pop_total,pop_males,pop_females,pop_native_total,pop_native_males,pop_native_females,pop_foreign_total,pop_foreign_males,pop_foreign_females,mig_total,mig_total_males,mig_total_females,mig_inflows,mig_inflows_males,mig_inflows_females,mig_outflows,mig_outflows_males,mig_outflows_females,income_tax_base,income_tax_liability,log_prices,price_growth_qoq_pct,price_growth_yoy_pct,log_price_growth_qoq,log_price_growth_yoy,contracts_per_10k,contracts_per_10k_qoq_pct,contracts_per_10k_yoy_pct,rent_to_income_proxy,annualized_rent_to_income_proxy,rent_to_tax_liability_proxy,outcome_price_growth,outcome_market_activity
0,00005,NaN,,2015,Q1,1,8061,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,,2015,Q2,2,8062,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,,2015,Q3,3,8063,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,,2015,Q4,4,8064,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,,2011,Q1,1,8045,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:

panel = panel.copy()
panel.columns = [str(c).strip() for c in panel.columns]

required_cols = [
    "Code", "Municipality", "Year", "Quarter",
    "Prices", "Contracts",
    "pop_total", "pop_native_total", "pop_foreign_total",
    "mig_inflows", "mig_outflows",
    "income_tax_base",
]
missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise ValueError(f"Missing required columns in analysis-ready panel: {missing_required}")

# Prefer the fixed rent-stress proxy if present; otherwise compute it safely later.
if "rent_to_income_proxy" not in panel.columns:
    panel["rent_to_income_proxy"] = np.nan

# Grouping
if "ever_treated" in panel.columns:
    panel["Group"] = np.where(pd.to_numeric(panel["ever_treated"], errors="coerce").fillna(0).astype(int) == 1, "Treated", "Control")
elif {"Treated 1", "Treated 2"}.issubset(panel.columns):
    ever_treated = (
        pd.to_numeric(panel["Treated 1"], errors="coerce").fillna(0).astype(int) == 1
    ) | (
        pd.to_numeric(panel["Treated 2"], errors="coerce").fillna(0).astype(int) == 1
    )
    panel["Group"] = np.where(ever_treated, "Treated", "Control")
else:
    raise ValueError("Could not identify treated vs control groups from the loaded panel.")

# Time
quarter_num_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
panel["Year"] = pd.to_numeric(panel["Year"], errors="coerce").astype("Int64")
panel["quarter_num"] = panel["Quarter"].map(quarter_num_map).astype("Int64")

bad_quarters = sorted(set(panel.loc[panel["quarter_num"].isna(), "Quarter"].dropna().tolist()))
if bad_quarters:
    raise ValueError(f"Unexpected quarter labels found: {bad_quarters}")

panel["time_index"] = panel["Year"].astype(int) * 4 + panel["quarter_num"].astype(int)

POST_START_YEAR = 2024
POST_START_QUARTER = "Q2"
POST_START_INDEX = POST_START_YEAR * 4 + quarter_num_map[POST_START_QUARTER]

panel["is_pre"] = panel["time_index"] < POST_START_INDEX
panel["is_post"] = panel["time_index"] >= POST_START_INDEX

# Numeric coercion
numeric_cols = [
    "Prices", "Contracts",
    "pop_total", "pop_native_total", "pop_foreign_total",
    "mig_inflows", "mig_outflows",
    "income_tax_base", "rent_to_income_proxy",
    "price_growth_yoy_pct",
]
for c in numeric_cols:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

# Flag coercion with reasonable fallbacks
flag_cols = [
    "price_observed_raw", "contracts_observed_raw",
    "price_imputed", "contracts_imputed",
    "has_any_observed_price", "has_any_observed_contracts",
    "exclude_from_price_analysis", "exclude_from_contract_analysis",
]
for c in flag_cols:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce").fillna(0).astype(int)

if "exclude_from_price_analysis" not in panel.columns:
    # Conservative fallback: if a municipality never has a positive finite price, exclude from price analysis
    tmp = (
        panel.assign(_valid_positive_price=np.isfinite(panel["Prices"]) & (panel["Prices"] > 0))
             .groupby("Code")["_valid_positive_price"].max()
             .astype(int)
             .rename("has_any_observed_price_fallback")
             .reset_index()
    )
    panel = panel.merge(tmp, on="Code", how="left")
    panel["exclude_from_price_analysis"] = (panel["has_any_observed_price_fallback"] == 0).astype(int)
    panel["has_any_observed_price"] = (panel["has_any_observed_price_fallback"] == 1).astype(int)
    panel = panel.drop(columns=["has_any_observed_price_fallback"])
else:
    if "has_any_observed_price" not in panel.columns:
        panel["has_any_observed_price"] = (panel["exclude_from_price_analysis"] == 0).astype(int)

if "exclude_from_contract_analysis" not in panel.columns:
    panel["exclude_from_contract_analysis"] = 0

if "has_any_observed_contracts" not in panel.columns:
    tmp = (
        panel.assign(_valid_contract=np.isfinite(panel["Contracts"]))
             .groupby("Code")["_valid_contract"].max()
             .astype(int)
             .rename("has_any_observed_contracts_fallback")
             .reset_index()
    )
    panel = panel.merge(tmp, on="Code", how="left")
    panel["has_any_observed_contracts"] = panel["has_any_observed_contracts_fallback"].fillna(0).astype(int)
    panel = panel.drop(columns=["has_any_observed_contracts_fallback"])

# Safe price/growth cleanup
panel.loc[~np.isfinite(panel["Prices"]) | (panel["Prices"] <= 0), "Prices"] = np.nan

if "price_growth_yoy_pct" in panel.columns:
    panel["price_growth_yoy_pct"] = pd.to_numeric(panel["price_growth_yoy_pct"], errors="coerce")
    panel.loc[~np.isfinite(panel["price_growth_yoy_pct"]), "price_growth_yoy_pct"] = np.nan
else:
    panel["price_growth_yoy_pct"] = (
        panel.sort_values(["Code", "Year", "quarter_num"])
             .groupby("Code")["Prices"]
             .pct_change(4) * 100
    )
    panel.loc[~np.isfinite(panel["price_growth_yoy_pct"]), "price_growth_yoy_pct"] = np.nan

# Recompute rent/income proxy safely so rows with invalid prices stay missing
panel["rent_to_income_proxy"] = np.where(
    (panel["Prices"].notna()) & np.isfinite(panel["Prices"]) & (panel["income_tax_base"] > 0),
    panel["Prices"] / panel["income_tax_base"],
    np.nan,
)

panel = panel.sort_values(["Code", "Year", "quarter_num"]).reset_index(drop=True)

display(panel[[
    "Code", "Municipality", "Year", "Quarter", "Group",
    "Prices", "price_growth_yoy_pct",
    "exclude_from_price_analysis", "has_any_observed_price"
]].head(12))


,Code,Municipality,Year,Quarter,Group,Prices,price_growth_yoy_pct,exclude_from_price_analysis,has_any_observed_price
0,00005,NaN,2015,Q1,Control,NaN,NaN,1,0
1,00005,NaN,2015,Q2,Control,NaN,NaN,1,0
2,00005,NaN,2015,Q3,Control,NaN,NaN,1,0
3,00005,NaN,2015,Q4,Control,NaN,NaN,1,0
4,02050,NaN,2011,Q1,Control,NaN,NaN,1,0
5,02050,NaN,2011,Q2,Control,NaN,NaN,1,0
6,02050,NaN,2011,Q3,Control,NaN,NaN,1,0
7,02050,NaN,2011,Q4,Control,NaN,NaN,1,0
8,02050,NaN,2012,Q1,Control,NaN,NaN,1,0
9,02050,NaN,2012,Q2,Control,NaN,NaN,1,0


## Table construction strategy

For continuous variables, the table uses **equal municipality weighting**:

1. First compute each municipality's mean over the relevant sample window.
2. Then average those municipality-level means within `Treated` and `Control`.

This prevents municipalities with more valid rows from receiving extra weight.

For counts:
- `Number of Municipalities` is the number of unique municipalities in each group.
- `Municipalities with Data on Rents` is the number of unique municipalities with at least one genuine observed rent-price series.


In [ ]:

# Masks for valid analysis samples
valid_price_mask = panel["exclude_from_price_analysis"].fillna(0).astype(int) == 0
valid_contract_mask = panel["exclude_from_contract_analysis"].fillna(0).astype(int) == 0

def municipality_mean_by_group(df: pd.DataFrame, value_col: str, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    tmp = tmp[["Code", "Group", value_col]].copy()
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")
    tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=[value_col])

    if tmp.empty:
        return pd.Series({"Treated": np.nan, "Control": np.nan}, dtype=float)

    muni_means = (
        tmp.groupby(["Code", "Group"], as_index=False)[value_col]
           .mean()
    )
    group_means = muni_means.groupby("Group")[value_col].mean()

    return pd.Series({
        "Treated": group_means.get("Treated", np.nan),
        "Control": group_means.get("Control", np.nan),
    })

def municipality_count_by_group(df: pd.DataFrame, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    counts = tmp.groupby("Group")["Code"].nunique()
    return pd.Series({
        "Treated": counts.get("Treated", 0),
        "Control": counts.get("Control", 0),
    })

def municipality_count_with_valid_rent_series(df: pd.DataFrame) -> pd.Series:
    muni = (
        df.groupby(["Code", "Group"], as_index=False)
          .agg(has_rent_data=("has_any_observed_price", "max"))
    )
    muni = muni[muni["has_rent_data"] == 1]
    counts = muni.groupby("Group")["Code"].nunique()
    return pd.Series({
        "Treated": counts.get("Treated", 0),
        "Control": counts.get("Control", 0),
    })

stats_rows = {
    "Average Rent Price Pre": municipality_mean_by_group(
        panel, "Prices", valid_price_mask & panel["is_pre"]
    ),
    "Average Rent Price Post": municipality_mean_by_group(
        panel, "Prices", valid_price_mask & panel["is_post"]
    ),
    "Yearly Rent Growth (2011-2025)": municipality_mean_by_group(
        panel,
        "price_growth_yoy_pct",
        valid_price_mask & panel["Year"].between(2011, 2025),
    ),
    "Yearly Rent Growth (2022-2024)": municipality_mean_by_group(
        panel,
        "price_growth_yoy_pct",
        valid_price_mask & panel["Year"].between(2022, 2024),
    ),
    "Average Rent Growth Post": municipality_mean_by_group(
        panel,
        "price_growth_yoy_pct",
        valid_price_mask & panel["is_post"],
    ),
    "Tenancy Agreements Pre": municipality_mean_by_group(
        panel, "Contracts", valid_contract_mask & panel["is_pre"]
    ),
    "Tenancy Agreements Post": municipality_mean_by_group(
        panel, "Contracts", valid_contract_mask & panel["is_post"]
    ),
    "Rent Price/Taxable Base": municipality_mean_by_group(
        panel, "rent_to_income_proxy", valid_price_mask
    ),
    "Yearly Taxable Base": municipality_mean_by_group(panel, "income_tax_base"),
    "Population": municipality_mean_by_group(panel, "pop_total"),
    "Natives": municipality_mean_by_group(panel, "pop_native_total"),
    "Foreigners": municipality_mean_by_group(panel, "pop_foreign_total"),
    "Net Inflows": municipality_mean_by_group(panel, "mig_inflows"),
    "Net Outflows": municipality_mean_by_group(panel, "mig_outflows"),
    "Number of Municipalities": municipality_count_by_group(panel),
    "Municipalities with Data on Rents": municipality_count_with_valid_rent_series(panel),
}

desc_table = pd.DataFrame(stats_rows).T.reset_index().rename(columns={"index": "Statistic"})
desc_table = desc_table[["Statistic", "Treated", "Control"]]
display(desc_table)

,Statistic,Treated,Control
0,Average Rent Price Pre,555.9226,459.2492
1,Average Rent Price Post,718.6952,556.2429
2,Yearly Rent Growth (2011-2025),3.1745,2.5673
3,Yearly Rent Growth (2022-2024),6.0793,4.2416
4,Average Rent Growth Post,2.9524,3.6889
5,Tenancy Agreements Pre,129.7083,2.7554
6,Tenancy Agreements Post,94.5836,2.7037
7,Rent Price/Taxable Base,0.0248,0.0226
8,Yearly Taxable Base,"22,847.3680","19,749.9643"
9,Population,"25,624.9658","1,062.9853"


In [ ]:

count_rows = {
    "Number of Municipalities",
    "Municipalities with Data on Rents",
}

desc_table_display = desc_table.copy()
for col in ["Treated", "Control"]:
    desc_table_display[col] = desc_table_display.apply(
        lambda r: int(round(r[col])) if r["Statistic"] in count_rows and pd.notna(r[col]) else r[col],
        axis=1,
    )

display(desc_table_display)

,Statistic,Treated,Control
0,Average Rent Price Pre,555.9226,459.2492
1,Average Rent Price Post,718.6952,556.2429
2,Yearly Rent Growth (2011-2025),3.1745,2.5673
3,Yearly Rent Growth (2022-2024),6.0793,4.2416
4,Average Rent Growth Post,2.9524,3.6889
5,Tenancy Agreements Pre,129.7083,2.7554
6,Tenancy Agreements Post,94.5836,2.7037
7,Rent Price/Taxable Base,0.0248,0.0226
8,Yearly Taxable Base,"22,847.3680","19,749.9643"
9,Population,"25,624.9658","1,062.9853"


In [7]:

OUT_CSV = OUT_DIR / "descriptive_stats_treated_control_fixed.csv"
OUT_XLSX = OUT_DIR / "descriptive_stats_treated_control_fixed.xlsx"
OUT_LATEX = OUT_DIR / "descriptive_stats_treated_control_fixed.tex"

desc_table_display.to_csv(OUT_CSV, index=False)
desc_table_display.to_excel(OUT_XLSX, index=False)

latex_table = desc_table_display.to_latex(index=False, escape=False, float_format="%.4f")
OUT_LATEX.write_text(latex_table, encoding="utf-8")

print("Saved:")
print(" -", OUT_CSV)
print(" -", OUT_XLSX)
print(" -", OUT_LATEX)

Saved:
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_fixed.csv
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_fixed.xlsx
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_fixed.tex


In [6]:

print("Unique municipalities by group:")
display(
    panel.groupby("Group")["Code"].nunique().rename("n_municipalities").reset_index()
)

print("Municipalities with genuine rent data by group:")
display(
    panel.groupby(["Group", "Code"], as_index=False)["has_any_observed_price"].max()
         .groupby("Group", as_index=False)["has_any_observed_price"].sum()
         .rename(columns={"has_any_observed_price": "n_municipalities_with_rent_data"})
)

print("Share of rows with valid price-analysis eligibility by group:")
display(
    panel.assign(valid_price_analysis=valid_price_mask.astype(int))
         .groupby("Group", as_index=False)["valid_price_analysis"]
         .mean()
)

print("Share of non-missing finite Prices within valid price-analysis sample by group:")
display(
    panel.loc[valid_price_mask].assign(has_price=panel.loc[valid_price_mask, "Prices"].notna().astype(int))
         .groupby("Group", as_index=False)["has_price"]
         .mean()
)

Unique municipalities by group:


,Group,n_municipalities
0,Control,679
1,Treated,271


Municipalities with genuine rent data by group:


,Group,n_municipalities_with_rent_data
0,Control,257
1,Treated,271


Share of rows with valid price-analysis eligibility by group:


,Group,valid_price_analysis
0,Control,0.3808
1,Treated,1.0000


Share of non-missing finite Prices within valid price-analysis sample by group:


,Group,has_price
0,Control,0.9961
1,Treated,1.0000
